In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## Load Data

In [ ]:
data_dir = Path('../data/input')

def load_cds_sheet(filepath):
    raw = pd.read_excel(filepath, header=None)

    first_col = raw.iloc[:, 0].astype(str).str.strip()
    name_rows = raw.index[first_col.eq('Name')]
    if len(name_rows) == 0:
        raise ValueError(f"Could not find 'Name' row in {filepath.name}")

    name_row = int(name_rows[0])
    data_start = name_row + 2

    company_names = raw.iloc[name_row, 2:].astype(str).str.strip()

    dates = pd.to_datetime(raw.iloc[data_start:, 0], errors='coerce')
    values = raw.iloc[data_start:, 2:].apply(pd.to_numeric, errors='coerce')

    df = pd.DataFrame(values.to_numpy(), index=dates, columns=company_names.values)
    df = df.loc[~df.index.isna()].sort_index()
    df = df.loc[:, df.columns.notna()]
    return df

def drop_duplicate_issuer_columns(df):
    dup_mask = pd.Index(df.columns).duplicated(keep='first')
    removed = int(dup_mask.sum())
    if removed > 0:
        df = df.loc[:, ~dup_mask]
    return df, removed

cds_1y = load_cds_sheet(data_dir / 'CDS_1y_mat_data.xlsx')
cds_3y = load_cds_sheet(data_dir / 'CDS_3y_mat_data.xlsx')
cds_5y = load_cds_sheet(data_dir / 'CDS_5y_mat_data.xlsx')

cds_1y, dup_1y = drop_duplicate_issuer_columns(cds_1y)
cds_3y, dup_3y = drop_duplicate_issuer_columns(cds_3y)
cds_5y, dup_5y = drop_duplicate_issuer_columns(cds_5y)

analysis_start_date = pd.Timestamp('2017-01-01')
cds_1y = cds_1y.loc[cds_1y.index >= analysis_start_date]
cds_3y = cds_3y.loc[cds_3y.index >= analysis_start_date]
cds_5y = cds_5y.loc[cds_5y.index >= analysis_start_date]

print(f"Analysis start date cutoff: {analysis_start_date.date()}")
print(f"Duplicate issuer columns removed -> 1Y: {dup_1y}, 3Y: {dup_3y}, 5Y: {dup_5y}")
print(f"\n1Y CDS Data Shape: {cds_1y.shape}")
print(f"3Y CDS Data Shape: {cds_3y.shape}")
print(f"5Y CDS Data Shape: {cds_5y.shape}")
print(f"\nDate Range: {cds_5y.index.min()} to {cds_5y.index.max()}")
print(f"\nCompanies (first 8): {list(cds_5y.columns[:8])}")

## Summary Statistics by Company and Maturity

In [ ]:
def calculate_summary_stats(df, maturity_name):
    stats = pd.DataFrame({
        'Maturity': maturity_name,
        'Company': df.columns,
        'Count': df.count().values,
        'Mean': df.mean().values,
        'Median': df.median().values,
        'Std': df.std().values,
        'Min': df.min().values,
        'Max': df.max().values,
        'Q25': df.quantile(0.25).values,
        'Q75': df.quantile(0.75).values,
        'Range': (df.max() - df.min()).values,
        'CV': (df.std() / df.mean()).values
    })
    return stats

stats_1y = calculate_summary_stats(cds_1y, '1Y')
stats_3y = calculate_summary_stats(cds_3y, '3Y')
stats_5y = calculate_summary_stats(cds_5y, '5Y')

all_stats = pd.concat([stats_1y, stats_3y, stats_5y], ignore_index=True)

print("Summary Statistics by Company and Maturity")
print("=" * 100)
all_stats

In [ ]:
print("\n" + "=" * 100)
print("1-YEAR MATURITY CDS SPREADS")
print("=" * 100)
display(stats_1y)

print("\n" + "=" * 100)
print("3-YEAR MATURITY CDS SPREADS")
print("=" * 100)
display(stats_3y)

print("\n" + "=" * 100)
print("5-YEAR MATURITY CDS SPREADS")
print("=" * 100)
display(stats_5y)

## Missing Data Analysis

In [ ]:
def analyze_missing_data(df, maturity_name):
    total_obs = len(df)
    
    missing_stats = pd.DataFrame({
        'Maturity': maturity_name,
        'Company': df.columns,
        'Total_Observations': total_obs,
        'NA_Count': df.isna().sum().values,
        'Valid_Count': df.notna().sum().values,
        'NA_Percentage': (df.isna().sum() / total_obs * 100).values,
        'Valid_Percentage': (df.notna().sum() / total_obs * 100).values
    })
    
    return missing_stats

missing_1y = analyze_missing_data(cds_1y, '1Y')
missing_3y = analyze_missing_data(cds_3y, '3Y')
missing_5y = analyze_missing_data(cds_5y, '5Y')

all_missing = pd.concat([missing_1y, missing_3y, missing_5y], ignore_index=True)

print("Missing Data Analysis by Company and Maturity")
print("=" * 100)
all_missing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (df, maturity, missing_df) in enumerate([
    (cds_1y, '1Y', missing_1y),
    (cds_3y, '3Y', missing_3y),
    (cds_5y, '5Y', missing_5y)
]):
    ax = axes[idx]
    missing_pct = missing_df.set_index('Company')['NA_Percentage']
    missing_pct.plot(kind='bar', ax=ax, color='coral')
    ax.set_title(f'Missing Data Percentage - {maturity} Maturity', fontsize=12, fontweight='bold')
    ax.set_xlabel('Company', fontsize=10)
    ax.set_ylabel('Missing Data (%)', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Sequential Repeat Analysis

In [ ]:
def count_sequential_repeats(series):
    clean_series = series.dropna()
    
    if len(clean_series) <= 1:
        return 0, 0, 0, 0
    
    changes = clean_series != clean_series.shift(1)
    groups = changes.cumsum()
    repeat_lengths = clean_series.groupby(groups).size()
    
    repeats = repeat_lengths[repeat_lengths > 1]
    total_repeats = (repeats - 1).sum()
    repeat_sequences = len(repeats)
    avg_repeat_length = repeats.mean() if len(repeats) > 0 else 0
    max_repeat_length = repeats.max() if len(repeats) > 0 else 0
    
    return total_repeats, repeat_sequences, avg_repeat_length, max_repeat_length

def analyze_sequential_repeats(df, maturity_name):
    repeat_stats = []
    
    for company in df.columns:
        total_rep, seq_count, avg_len, max_len = count_sequential_repeats(df[company])
        valid_obs = df[company].notna().sum()
        
        repeat_stats.append({
            'Maturity': maturity_name,
            'Company': company,
            'Total_Repeats': total_rep,
            'Repeat_Sequences': seq_count,
            'Avg_Repeat_Length': avg_len,
            'Max_Repeat_Length': max_len,
            'Valid_Observations': valid_obs,
            'Repeat_Percentage': (total_rep / valid_obs * 100) if valid_obs > 0 else 0
        })
    
    return pd.DataFrame(repeat_stats)

repeats_1y = analyze_sequential_repeats(cds_1y, '1Y')
repeats_3y = analyze_sequential_repeats(cds_3y, '3Y')
repeats_5y = analyze_sequential_repeats(cds_5y, '5Y')

all_repeats = pd.concat([repeats_1y, repeats_3y, repeats_5y], ignore_index=True)

print("Sequential Repeat Analysis by Company and Maturity")
print("=" * 100)
all_repeats

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, (maturity, repeat_df) in enumerate([
    ('1Y', repeats_1y),
    ('3Y', repeats_3y),
    ('5Y', repeats_5y)
]):
    ax1 = axes[0, idx]
    repeat_pct = repeat_df.set_index('Company')['Repeat_Percentage']
    repeat_pct.plot(kind='bar', ax=ax1, color='skyblue')
    ax1.set_title(f'Repeat Percentage - {maturity}', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Company', fontsize=10)
    ax1.set_ylabel('Repeat %', fontsize=10)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    ax2 = axes[1, idx]
    avg_len = repeat_df.set_index('Company')['Avg_Repeat_Length']
    avg_len.plot(kind='bar', ax=ax2, color='lightgreen')
    ax2.set_title(f'Avg Repeat Length - {maturity}', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Company', fontsize=10)
    ax2.set_ylabel('Days', fontsize=10)
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Liquidity Metrics

In [ ]:
def calculate_liquidity_metrics(df, maturity_name):
    liquidity_metrics = []
    
    for company in df.columns:
        series = df[company]
        total_obs = len(series)
        valid_obs = series.notna().sum()
        
        data_availability = (valid_obs / total_obs * 100) if total_obs > 0 else 0
        
        clean_series = series.dropna()
        if len(clean_series) > 1:
            changes = (clean_series != clean_series.shift(1)).sum() - 1
            change_frequency = (changes / (len(clean_series) - 1) * 100) if len(clean_series) > 1 else 0
        else:
            change_frequency = 0
        
        _, _, avg_len, max_len = count_sequential_repeats(series)
        
        change_score = min(change_frequency, 100) * 0.4
        availability_score = data_availability * 0.3
        
        if avg_len > 0:
            repeat_score = max(0, 30 - (avg_len - 1) * 5)
        else:
            repeat_score = 30
        
        liquidity_score = change_score + availability_score + repeat_score
        
        liquidity_metrics.append({
            'Maturity': maturity_name,
            'Company': company,
            'Data_Availability_%': data_availability,
            'Change_Frequency_%': change_frequency,
            'Avg_Repeat_Length': avg_len,
            'Max_Repeat_Length': max_len,
            'Change_Score': change_score,
            'Availability_Score': availability_score,
            'Repeat_Score': repeat_score,
            'Liquidity_Score': liquidity_score
        })
    
    return pd.DataFrame(liquidity_metrics)

liquidity_1y = calculate_liquidity_metrics(cds_1y, '1Y')
liquidity_3y = calculate_liquidity_metrics(cds_3y, '3Y')
liquidity_5y = calculate_liquidity_metrics(cds_5y, '5Y')

all_liquidity = pd.concat([liquidity_1y, liquidity_3y, liquidity_5y], ignore_index=True)

print("Liquidity Metrics by Company and Maturity")
print("=" * 100)
all_liquidity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (maturity, liq_df) in enumerate([
    ('1Y', liquidity_1y),
    ('3Y', liquidity_3y),
    ('5Y', liquidity_5y)
]):
    ax = axes[idx]
    liq_score = liq_df.set_index('Company')['Liquidity_Score']
    colors = plt.cm.RdYlGn(liq_score / 100)
    liq_score.plot(kind='bar', ax=ax, color=colors)
    ax.set_title(f'Liquidity Score - {maturity} Maturity', fontsize=12, fontweight='bold')
    ax.set_xlabel('Company', fontsize=10)
    ax.set_ylabel('Liquidity Score (0-100)', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='Threshold (50)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
liquidity_pivot = all_liquidity.pivot(index='Company', columns='Maturity', values='Liquidity_Score')
liquidity_pivot.plot(kind='bar', ax=ax1)
ax1.set_title('Liquidity Score Comparison Across Maturities', fontsize=14, fontweight='bold')
ax1.set_xlabel('Company', fontsize=12)
ax1.set_ylabel('Liquidity Score', fontsize=12)
ax1.legend(title='Maturity', fontsize=10)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
change_pivot = all_liquidity.pivot(index='Company', columns='Maturity', values='Change_Frequency_%')
change_pivot.plot(kind='bar', ax=ax2)
ax2.set_title('Change Frequency Comparison Across Maturities', fontsize=14, fontweight='bold')
ax2.set_xlabel('Company', fontsize=12)
ax2.set_ylabel('Change Frequency (%)', fontsize=12)
ax2.legend(title='Maturity', fontsize=10)
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Comprehensive Summary Report

In [ ]:
comprehensive_summary = all_stats.merge(
    all_missing[['Maturity', 'Company', 'NA_Count', 'NA_Percentage']], 
    on=['Maturity', 'Company']
).merge(
    all_repeats[['Maturity', 'Company', 'Total_Repeats', 'Avg_Repeat_Length', 'Repeat_Percentage']], 
    on=['Maturity', 'Company']
).merge(
    all_liquidity[['Maturity', 'Company', 'Change_Frequency_%', 'Liquidity_Score']], 
    on=['Maturity', 'Company']
)

column_order = [
    'Company', 'Maturity', 'Count', 'NA_Count', 'NA_Percentage',
    'Mean', 'Median', 'Std', 'Min', 'Max', 'Range', 'CV',
    'Total_Repeats', 'Avg_Repeat_Length', 'Repeat_Percentage',
    'Change_Frequency_%', 'Liquidity_Score'
]
comprehensive_summary = comprehensive_summary[column_order]

print("\n" + "=" * 100)
print("COMPREHENSIVE SUMMARY: ALL METRICS BY COMPANY AND MATURITY")
print("=" * 100)
comprehensive_summary

In [ ]:
output_dir = Path('../data/output')
output_dir.mkdir(exist_ok=True)

comprehensive_summary.to_csv(output_dir / 'cds_comprehensive_analysis.csv', index=False)
all_liquidity.to_csv(output_dir / 'cds_liquidity_metrics.csv', index=False)

print(f"Exported: {output_dir / 'cds_comprehensive_analysis.csv'}")
print(f"Exported: {output_dir / 'cds_liquidity_metrics.csv'}")

## Key Insights and Rankings

In [ ]:
print("\n" + "=" * 100)
print("LIQUIDITY RANKINGS BY MATURITY")
print("=" * 100)

for maturity in ['1Y', '3Y', '5Y']:
    print(f"\n{maturity} Maturity - Top to Bottom (by Liquidity Score):")
    print("-" * 60)
    maturity_data = all_liquidity[all_liquidity['Maturity'] == maturity].sort_values(
        'Liquidity_Score', ascending=False
    )[['Company', 'Liquidity_Score', 'Change_Frequency_%', 'Data_Availability_%']]
    display(maturity_data.reset_index(drop=True))

In [ ]:
print("\n" + "=" * 100)
print("AVERAGE METRICS BY MATURITY")
print("=" * 100)

maturity_summary = all_liquidity.groupby('Maturity').agg({
    'Data_Availability_%': 'mean',
    'Change_Frequency_%': 'mean',
    'Avg_Repeat_Length': 'mean',
    'Max_Repeat_Length': 'mean',
    'Liquidity_Score': 'mean'
}).round(2)

maturity_summary

In [ ]:
print("\n" + "=" * 100)
print("INSTRUMENTS REQUIRING ATTENTION")
print("=" * 100)

liquidity_threshold = 50
missing_threshold = 10
repeat_threshold = 20

problematic = comprehensive_summary[
    (comprehensive_summary['Liquidity_Score'] < liquidity_threshold) |
    (comprehensive_summary['NA_Percentage'] > missing_threshold) |
    (comprehensive_summary['Repeat_Percentage'] > repeat_threshold)
].sort_values('Liquidity_Score')

print(f"\nInstruments with Liquidity Score < {liquidity_threshold}, ")
print(f"Missing Data > {missing_threshold}%, or Repeat % > {repeat_threshold}%:")
print("-" * 100)
problematic[[
    'Company', 'Maturity', 'NA_Percentage', 'Repeat_Percentage', 
    'Change_Frequency_%', 'Liquidity_Score'
]]

## Heatmap Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

ax1 = axes[0, 0]
liq_heatmap = all_liquidity.pivot(index='Company', columns='Maturity', values='Liquidity_Score')
sns.heatmap(liq_heatmap, annot=False, cmap='RdYlGn', ax=ax1, cbar_kws={'label': 'Score'})
ax1.set_title('Liquidity Score by Company and Maturity', fontsize=12, fontweight='bold')

ax2 = axes[0, 1]
missing_heatmap = all_missing.pivot(index='Company', columns='Maturity', values='NA_Percentage')
sns.heatmap(missing_heatmap, annot=False, cmap='RdYlGn_r', ax=ax2, cbar_kws={'label': '%'})
ax2.set_title('Missing Data % by Company and Maturity', fontsize=12, fontweight='bold')

ax3 = axes[1, 0]
change_heatmap = all_liquidity.pivot(index='Company', columns='Maturity', values='Change_Frequency_%')
sns.heatmap(change_heatmap, annot=False, cmap='Blues', ax=ax3, cbar_kws={'label': '%'})
ax3.set_title('Change Frequency % by Company and Maturity', fontsize=12, fontweight='bold')

ax4 = axes[1, 1]
repeat_heatmap = all_liquidity.pivot(index='Company', columns='Maturity', values='Avg_Repeat_Length')
sns.heatmap(repeat_heatmap, annot=False, cmap='YlOrRd', ax=ax4, cbar_kws={'label': 'Days'})
ax4.set_title('Avg Repeat Length by Company and Maturity', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
screen = comprehensive_summary.copy()

liq_thr = 50
miss_thr = 10
rep_thr = 20

screen['Use_for_model'] = (
    (screen['Liquidity_Score'] >= liq_thr)
    & (screen['NA_Percentage'] <= miss_thr)
    & (screen['Repeat_Percentage'] <= rep_thr)
)

screen['High_confidence'] = (
    (screen['Liquidity_Score'] >= 70)
    & (screen['NA_Percentage'] <= 5)
    & (screen['Repeat_Percentage'] <= 15)
)

coverage = screen.groupby('Maturity').agg(
    total=('Company', 'count'),
    use_count=('Use_for_model', 'sum'),
    high_conf_count=('High_confidence', 'sum'),
    avg_liq=('Liquidity_Score', 'mean'),
    med_liq=('Liquidity_Score', 'median'),
    avg_missing=('NA_Percentage', 'mean'),
    avg_repeat=('Repeat_Percentage', 'mean')
)
coverage['use_pct'] = (coverage['use_count'] / coverage['total'] * 100).round(1)
coverage['high_conf_pct'] = (coverage['high_conf_count'] / coverage['total'] * 100).round(1)
coverage = coverage.round(2)

print('Tradability Screen by Maturity')
display(coverage)

print('\nLowest-liquidity names (10 worst overall):')
worst = screen.sort_values('Liquidity_Score').loc[:, [
    'Company', 'Maturity', 'Liquidity_Score', 'NA_Percentage', 'Repeat_Percentage', 'Change_Frequency_%'
]].head(10)
display(worst)

print('\nBest-liquidity names (10 best overall):')
best = screen.sort_values('Liquidity_Score', ascending=False).loc[:, [
    'Company', 'Maturity', 'Liquidity_Score', 'NA_Percentage', 'Repeat_Percentage', 'Change_Frequency_%'
]].head(10)
display(best)

print('\nCounts failing each criterion (by maturity):')
fails = screen.groupby('Maturity').apply(
    lambda g: pd.Series({
        'fail_liquidity_lt_50': (g['Liquidity_Score'] < liq_thr).sum(),
        'fail_missing_gt_10pct': (g['NA_Percentage'] > miss_thr).sum(),
        'fail_repeat_gt_20pct': (g['Repeat_Percentage'] > rep_thr).sum(),
    })
)
display(fails)

In [ ]:
for maturity, df in [('1Y', cds_1y), ('3Y', cds_3y), ('5Y', cds_5y)]:
    cols = pd.Series(df.columns.astype(str))
    unnamed = cols.str.startswith('Unnamed:').sum()
    print(f"{maturity}: {unnamed}/{len(cols)} columns are 'Unnamed:*' ({unnamed/len(cols)*100:.1f}%)")

In [ ]:
for nm in ['cds_1y', 'cds_3y', 'cds_5y']:
    df = globals()[nm]
    dup_mask = pd.Index(df.columns).duplicated(keep='first')
    dup_count = int(dup_mask.sum())
    if dup_count > 0:
        globals()[nm] = df.loc[:, ~dup_mask]
    print(f"{nm}: removed {dup_count} duplicate columns; now {globals()[nm].shape[1]} columns")

## Date-Level Liquidity Screening

In [ ]:
screen_cfg = {
    'window_obs': 60,
    'min_window_obs': 45,
    'min_availability': 0.90,
    'min_change_frequency': 0.25,
    'max_stale_days': 10,
    'min_segment_years': 3,
}

cfg_table = pd.DataFrame([
    {'Rule': 'window_obs', 'Value': screen_cfg['window_obs'], 'Meaning': 'Rolling window length (observations)'},
    {'Rule': 'min_window_obs', 'Value': screen_cfg['min_window_obs'], 'Meaning': 'Minimum observations required in rolling window'},
    {'Rule': 'min_availability', 'Value': screen_cfg['min_availability'], 'Meaning': 'Minimum non-missing ratio in rolling window'},
    {'Rule': 'min_change_frequency', 'Value': screen_cfg['min_change_frequency'], 'Meaning': 'Minimum fraction of non-zero day-to-day changes'},
    {'Rule': 'max_stale_days', 'Value': screen_cfg['max_stale_days'], 'Meaning': 'Maximum consecutive unchanged observed days'},
    {'Rule': 'min_segment_years', 'Value': screen_cfg['min_segment_years'], 'Meaning': 'Minimum contiguous good calendar years to keep segment'},
])

maturity_panels = {
    '1Y': cds_1y.loc[:, ~pd.Index(cds_1y.columns).duplicated(keep='first')],
    '3Y': cds_3y.loc[:, ~pd.Index(cds_3y.columns).duplicated(keep='first')],
    '5Y': cds_5y.loc[:, ~pd.Index(cds_5y.columns).duplicated(keep='first')],
}


def _stale_days(series):
    out = np.full(len(series), np.nan)
    last_val = np.nan
    streak = 0
    for i, val in enumerate(series.to_numpy()):
        if pd.isna(val):
            out[i] = np.nan
            continue
        if pd.isna(last_val) or val != last_val:
            streak = 0
        else:
            streak += 1
        out[i] = streak
        last_val = val
    return pd.Series(out, index=series.index)


def _series_date_flags(series, maturity, company, cfg):
    s = pd.to_numeric(series, errors='coerce').sort_index()
    valid = s.notna()

    changed = (s != s.shift(1)) & valid & s.shift(1).notna()

    rolling_valid = valid.astype(float).rolling(cfg['window_obs'], min_periods=cfg['min_window_obs']).mean()
    valid_count = valid.astype(float).rolling(cfg['window_obs'], min_periods=cfg['min_window_obs']).sum()
    change_count = changed.astype(float).rolling(cfg['window_obs'], min_periods=cfg['min_window_obs']).sum()
    rolling_change_freq = change_count / (valid_count - 1).clip(lower=1)

    stale_days = _stale_days(s)

    fail_missing = ~valid
    fail_low_avail = valid & (rolling_valid < cfg['min_availability'])
    fail_low_change = valid & (rolling_change_freq < cfg['min_change_frequency'])
    fail_stale = valid & (stale_days > cfg['max_stale_days'])

    good = valid & (~fail_low_avail) & (~fail_low_change) & (~fail_stale)

    reason = np.where(fail_missing, 'missing', '')
    reason = np.where((reason == '') & fail_low_avail, 'low_availability', reason)
    reason = np.where((reason == '') & fail_low_change, 'low_change_frequency', reason)
    reason = np.where((reason == '') & fail_stale, 'stale_quote', reason)
    reason = np.where(reason == '', 'good', reason)

    combo = (
        fail_missing.map({True: 'missing', False: ''}).astype(str)
        + fail_low_avail.map({True: '|low_availability', False: ''}).astype(str)
        + fail_low_change.map({True: '|low_change_frequency', False: ''}).astype(str)
        + fail_stale.map({True: '|stale_quote', False: ''}).astype(str)
    ).str.lstrip('|')
    combo = combo.replace('', 'good')

    return pd.DataFrame({
        'Date': s.index,
        'Maturity': maturity,
        'Company': company,
        'Spread': s.values,
        'Is_Valid': valid.values,
        'Rolling_Availability': rolling_valid.values,
        'Rolling_Change_Frequency': rolling_change_freq.values,
        'Stale_Days': stale_days.values,
        'Fail_Missing': fail_missing.values,
        'Fail_Low_Availability': fail_low_avail.values,
        'Fail_Low_Change': fail_low_change.values,
        'Fail_Stale': fail_stale.values,
        'Is_Good_Day': good.values,
        'Primary_Reason': reason,
        'Reason_Combo': combo.values,
    })


all_flags = []
for mty, panel in maturity_panels.items():
    for comp in panel.columns:
        all_flags.append(_series_date_flags(panel[comp], mty, comp, screen_cfg))

cds_date_flags = pd.concat(all_flags, ignore_index=True)


def _extract_segments(flags_df, min_segment_years):
    out = []
    min_days = int(min_segment_years * 365)
    for (mty, comp), g in flags_df.groupby(['Maturity', 'Company'], sort=False):
        gs = g.sort_values('Date').copy()
        block = gs['Is_Good_Day'].ne(gs['Is_Good_Day'].shift()).cumsum()
        gs['_block'] = block

        for _, seg in gs.groupby('_block'):
            is_good = bool(seg['Is_Good_Day'].iloc[0])
            start = seg['Date'].iloc[0]
            end = seg['Date'].iloc[-1]
            n_obs = len(seg)
            segment_days = int((end - start).days)
            if is_good:
                out.append({
                    'Maturity': mty,
                    'Company': comp,
                    'Segment_Start': start,
                    'Segment_End': end,
                    'Segment_Obs': n_obs,
                    'Segment_Days': segment_days,
                    'Keep_For_Model': segment_days >= min_days,
                })
    return pd.DataFrame(out)


good_segments = _extract_segments(cds_date_flags, screen_cfg['min_segment_years'])

series_quality = cds_date_flags.groupby(['Maturity', 'Company']).agg(
    Total_Obs=('Date', 'count'),
    Good_Obs=('Is_Good_Day', 'sum'),
    First_Date=('Date', 'min'),
    Last_Date=('Date', 'max'),
).reset_index()
series_quality['Good_Pct'] = (series_quality['Good_Obs'] / series_quality['Total_Obs'] * 100).round(2)

kept_segments = good_segments[good_segments['Keep_For_Model']].copy()
series_keep_summary = kept_segments.groupby(['Maturity', 'Company']).agg(
    Kept_Segments=('Keep_For_Model', 'count'),
    Kept_Obs=('Segment_Obs', 'sum'),
    Kept_Start=('Segment_Start', 'min'),
    Kept_End=('Segment_End', 'max'),
).reset_index()

series_quality = series_quality.merge(series_keep_summary, on=['Maturity', 'Company'], how='left')
series_quality['Kept_Segments'] = series_quality['Kept_Segments'].fillna(0).astype(int)
series_quality['Kept_Obs'] = series_quality['Kept_Obs'].fillna(0).astype(int)
series_quality['Kept_Pct'] = (series_quality['Kept_Obs'] / series_quality['Total_Obs'] * 100).round(2)
series_quality['Exclude_All'] = series_quality['Kept_Obs'].eq(0)

print('Date-level screening configuration:')
display(cfg_table)
print('\nPer-series keep summary (preview):')
display(series_quality.sort_values(['Exclude_All', 'Kept_Pct', 'Good_Pct'], ascending=[False, True, True]).head(15))

In [ ]:
reason_breakdown = (
    cds_date_flags[~cds_date_flags['Is_Good_Day']]
    .groupby(['Maturity', 'Primary_Reason'])
    .size()
    .rename('Excluded_Days')
    .reset_index()
)

exclude_series = series_quality[series_quality['Exclude_All']].sort_values(['Maturity', 'Company'])
partial_series = series_quality[(~series_quality['Exclude_All']) & (series_quality['Kept_Pct'] < 100)].sort_values(['Maturity', 'Kept_Pct'])
full_series = series_quality[series_quality['Kept_Pct'] == 100].sort_values(['Maturity', 'Company'])

keep_windows = kept_segments.sort_values(['Maturity', 'Company', 'Segment_Start'])

final_date_panel = cds_date_flags.merge(
    keep_windows[['Maturity', 'Company', 'Segment_Start', 'Segment_End']],
    on=['Maturity', 'Company'],
    how='left'
)
final_date_panel['In_Kept_Window'] = (
    (final_date_panel['Date'] >= final_date_panel['Segment_Start'])
    & (final_date_panel['Date'] <= final_date_panel['Segment_End'])
)
final_date_panel['In_Kept_Window'] = final_date_panel['In_Kept_Window'].fillna(False)
final_date_panel['Use_For_Model'] = final_date_panel['Is_Good_Day'] & final_date_panel['In_Kept_Window']

final_date_panel = (
    final_date_panel
    .groupby(['Date', 'Maturity', 'Company', 'Spread', 'Is_Valid', 'Rolling_Availability',
              'Rolling_Change_Frequency', 'Stale_Days', 'Fail_Missing', 'Fail_Low_Availability',
              'Fail_Low_Change', 'Fail_Stale', 'Is_Good_Day', 'Primary_Reason', 'Reason_Combo'], as_index=False)
    .agg(Use_For_Model=('Use_For_Model', 'max'))
)

print('Excluded entire series (no sufficiently long liquid window):')
display(exclude_series[['Maturity', 'Company', 'Good_Pct', 'Kept_Pct', 'Good_Obs', 'Kept_Obs']])

print('\nPartially usable series (use only selected date windows):')
display(partial_series[['Maturity', 'Company', 'Good_Pct', 'Kept_Pct', 'Kept_Segments', 'Kept_Start', 'Kept_End']].head(25))

print('\nExact kept windows (first 40 rows):')
display(keep_windows.head(40))

print('\nReason breakdown for excluded days:')
display(reason_breakdown)

output_dir = Path('../data/output')
output_dir.mkdir(exist_ok=True)

cfg_table.to_csv(output_dir / 'cds_liquidity_screen_config.csv', index=False)
series_quality.to_csv(output_dir / 'cds_series_quality_screen.csv', index=False)
keep_windows.to_csv(output_dir / 'cds_keep_windows_by_series.csv', index=False)
exclude_series.to_csv(output_dir / 'cds_exclude_series.csv', index=False)
reason_breakdown.to_csv(output_dir / 'cds_exclusion_reason_breakdown.csv', index=False)
final_date_panel.to_csv(output_dir / 'cds_date_level_model_use_flag.csv', index=False)

print(f"\nExported: {output_dir / 'cds_liquidity_screen_config.csv'}")
print(f"Exported: {output_dir / 'cds_series_quality_screen.csv'}")
print(f"Exported: {output_dir / 'cds_keep_windows_by_series.csv'}")
print(f"Exported: {output_dir / 'cds_exclude_series.csv'}")
print(f"Exported: {output_dir / 'cds_exclusion_reason_breakdown.csv'}")
print(f"Exported: {output_dir / 'cds_date_level_model_use_flag.csv'}")

In [ ]:
series_profile = series_quality[['Maturity', 'Company', 'Total_Obs', 'Good_Obs', 'Good_Pct', 'Kept_Obs', 'Kept_Pct', 'Kept_Segments', 'Kept_Start', 'Kept_End', 'First_Date', 'Last_Date']].copy()

series_profile['Pattern'] = np.select(
    [
        series_profile['Kept_Obs'].eq(0),
        (series_profile['Kept_Start'] <= series_profile['First_Date']) & (series_profile['Kept_End'] < series_profile['Last_Date']),
        (series_profile['Kept_Start'] > series_profile['First_Date']) & (series_profile['Kept_End'] >= series_profile['Last_Date']),
        (series_profile['Kept_Start'] > series_profile['First_Date']) & (series_profile['Kept_End'] < series_profile['Last_Date']),
        (series_profile['Kept_Start'] <= series_profile['First_Date']) & (series_profile['Kept_End'] >= series_profile['Last_Date']),
    ],
    [
        'Exclude_All',
        'Good_Early_Then_Degrade',
        'Bad_Early_Then_Good',
        'Good_Middle_Only',
        'Good_Most_Of_Sample',
    ],
    default='Mixed'
)

pattern_summary = series_profile.groupby(['Maturity', 'Pattern']).size().rename('Series_Count').reset_index()

print('Pattern summary (how usability evolves over time):')
display(pattern_summary.sort_values(['Maturity', 'Series_Count'], ascending=[True, False]))

print('\nExamples: good early then degrades (first 15):')
display(series_profile[series_profile['Pattern'] == 'Good_Early_Then_Degrade']
        .sort_values(['Maturity', 'Kept_End'])
        .head(15))

print('\nExamples: bad early then good (first 15):')
display(series_profile[series_profile['Pattern'] == 'Bad_Early_Then_Good']
        .sort_values(['Maturity', 'Kept_Start'])
        .head(15))

print('\nExamples: good middle only (first 15):')
display(series_profile[series_profile['Pattern'] == 'Good_Middle_Only']
        .sort_values(['Maturity', 'Kept_Pct'])
        .head(15))

In [ ]:
exclude_counts = series_quality.groupby('Maturity').agg(
    total_series=('Company', 'count'),
    fully_excluded=('Exclude_All', 'sum'),
    partially_used=('Kept_Pct', lambda x: ((x > 0) & (x < 100)).sum()),
    fully_used=('Kept_Pct', lambda x: (x == 100).sum()),
).reset_index()
exclude_counts['fully_excluded_pct'] = (exclude_counts['fully_excluded'] / exclude_counts['total_series'] * 100).round(1)

print('Series inclusion counts by maturity:')
display(exclude_counts)

print('\nTop 10 earliest cutoff dates (good early then degrades):')
early_cut = (
    series_profile[series_profile['Pattern'] == 'Good_Early_Then_Degrade']
    .sort_values('Kept_End')
    [['Maturity', 'Company', 'Kept_Start', 'Kept_End', 'Kept_Pct']]
    .head(10)
)
display(early_cut)

print('\nTop 10 latest start dates (bad early then good):')
late_start = (
    series_profile[series_profile['Pattern'] == 'Bad_Early_Then_Good']
    .sort_values('Kept_Start', ascending=False)
    [['Maturity', 'Company', 'Kept_Start', 'Kept_End', 'Kept_Pct']]
    .head(10)
)
display(late_start)

In [ ]:
cds_model_panel_long = (
    final_date_panel.loc[
        final_date_panel['Use_For_Model'],
        ['Date', 'Maturity', 'Company', 'Spread', 'Rolling_Availability', 'Rolling_Change_Frequency', 'Stale_Days']
    ]
    .sort_values(['Date', 'Maturity', 'Company'])
    .reset_index(drop=True)
)

cds_model_panel_wide = {}
for mty, g in cds_model_panel_long.groupby('Maturity'):
    cds_model_panel_wide[mty] = g.pivot(index='Date', columns='Company', values='Spread').sort_index()

coverage_final = (
    cds_model_panel_long.groupby('Maturity')
    .agg(
        kept_rows=('Spread', 'size'),
        kept_firms=('Company', 'nunique'),
        start=('Date', 'min'),
        end=('Date', 'max')
    )
    .reset_index()
)

print('Final model-ready panel coverage:')
display(coverage_final)

print('\nPreview of long panel:')
display(cds_model_panel_long.head(20))

In [ ]:
cutoff_date = pd.Timestamp('2017-01-01')

min_date_in_panel = cds_model_panel_long['Date'].min()
if min_date_in_panel < cutoff_date:
    raise ValueError(f"Found dates before cutoff in final panel: {min_date_in_panel}")

cds_model_panel_long_post2017 = cds_model_panel_long.copy()

coverage_post2017 = (
    cds_model_panel_long_post2017.groupby('Maturity')
    .agg(
        kept_rows=('Spread', 'size'),
        kept_firms=('Company', 'nunique'),
        start=('Date', 'min'),
        end=('Date', 'max')
    )
    .reset_index()
)

print('Post-2017 model-ready panel coverage (cutoff applied at load stage):')
display(coverage_post2017)

filter_dir = Path('../data/cds_filters')
filter_dir.mkdir(parents=True, exist_ok=True)
gvkey_maturity_windows_path = filter_dir / 'gvkey_maturity_simulation_windows.csv'
cleaned_cds_path = filter_dir / 'cds_clean_for_comparison.csv'

for old_file in filter_dir.glob('*.csv'):
    if old_file.name not in {gvkey_maturity_windows_path.name, cleaned_cds_path.name}:
        try:
            old_file.unlink()
        except PermissionError:
            print(f"WARNING: Could not remove locked file {old_file.name}")

import re

def _normalize_company_name(name):
    legal = {
        'SA','SPA','SE','AG','NV','OYJ','GROUP','HOLDING','HOLDINGS','PLC','INC','CORP','CORPORATION',
        'CO','COMPANY','LTD','LIMITED','REINSURANCE','N','V','S','P','A'
    }
    s = str(name).upper()
    s = re.sub(r'\bSNR.*$', '', s)
    s = re.sub(r'\bSUB.*$', '', s)
    s = re.sub(r'\bMM\d+.*$', '', s)
    s = re.sub(r'\bCDS\b.*$', '', s)
    s = s.replace('PREM', ' ').replace('MID', ' ')
    s = re.sub(r'[^A-Z0-9 ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    toks = [t for t in s.split() if t and t not in legal and not t.isdigit() and len(t) > 1]
    return ' '.join(toks), tuple(sorted(set(toks)))

merton_map_path = Path('../data/output/merged_data_with_merton.csv')
if not merton_map_path.exists():
    raise FileNotFoundError(f"Required file not found for gvkey mapping: {merton_map_path}")

allowed_company_dates = (
    final_date_panel.loc[final_date_panel['Use_For_Model'], ['Date', 'Maturity', 'Company', 'Spread']]
    .rename(columns={'Date': 'date', 'Company': 'cds_company'})
    .copy()
)
allowed_company_dates['date'] = pd.to_datetime(allowed_company_dates['date'], errors='coerce').dt.normalize()
allowed_company_dates = allowed_company_dates[
    allowed_company_dates['date'].notna()
    & allowed_company_dates['cds_company'].notna()
].drop_duplicates()

merton_map = pd.read_csv(merton_map_path, usecols=['company', 'gvkey'])
merton_map = merton_map[merton_map['company'].notna() & merton_map['gvkey'].notna()].drop_duplicates().copy()

cds_names = pd.DataFrame({'cds_company': sorted(allowed_company_dates['cds_company'].unique())})
cds_names[['norm_name', 'norm_tokens']] = cds_names['cds_company'].apply(lambda x: pd.Series(_normalize_company_name(x)))

merton_names = merton_map[['company', 'gvkey']].drop_duplicates().copy()
merton_names[['norm_name', 'norm_tokens']] = merton_names['company'].apply(lambda x: pd.Series(_normalize_company_name(x)))

exact = cds_names.merge(
    merton_names[['company', 'gvkey', 'norm_name']].drop_duplicates(),
    on='norm_name',
    how='left'
)

fallback_rows = []
unmatched_exact = exact[exact['gvkey'].isna()][['cds_company', 'norm_tokens']].copy()
for _, row in unmatched_exact.iterrows():
    tok_set = set(row['norm_tokens'])
    if not tok_set:
        continue
    best = None
    for _, cand in merton_names.iterrows():
        cand_set = set(cand['norm_tokens'])
        if not cand_set:
            continue
        inter = len(tok_set & cand_set)
        if inter == 0:
            continue
        score = inter / max(len(tok_set), len(cand_set))
        if best is None or score > best[0]:
            best = (score, cand['company'], cand['gvkey'])
    if best is not None and best[0] >= 0.50:
        fallback_rows.append({
            'cds_company': row['cds_company'],
            'mapped_company': best[1],
            'gvkey': best[2],
            'match_score': best[0],
            'match_type': 'token_overlap'
        })

fallback = pd.DataFrame(fallback_rows) if fallback_rows else pd.DataFrame(
    columns=['cds_company', 'mapped_company', 'gvkey', 'match_score', 'match_type']
)

mapped_exact = exact[exact['gvkey'].notna()][['cds_company', 'company', 'gvkey']].drop_duplicates()
mapped_exact = mapped_exact.rename(columns={'company': 'mapped_company'})
mapped_exact['match_type'] = 'exact_norm'
mapped_exact['match_score'] = 1.0

mapped = pd.concat([mapped_exact, fallback], ignore_index=True)
mapped = mapped.drop_duplicates(subset=['cds_company'], keep='first')

cleaned_cds = (
    allowed_company_dates
    .merge(mapped[['cds_company', 'mapped_company', 'gvkey', 'match_type', 'match_score']], on='cds_company', how='inner')
    [['date', 'Maturity', 'cds_company', 'mapped_company', 'gvkey', 'Spread', 'match_type', 'match_score']]
    .rename(columns={'Maturity': 'maturity', 'Spread': 'spread'})
    .drop_duplicates()
    .sort_values(['date', 'gvkey', 'maturity'])
)

maturity_to_horizon = {'1Y': 252, '3Y': 756, '5Y': 1260}
gvkey_maturity_windows = (
    cleaned_cds.assign(maturity=lambda d: d['maturity'].astype(str).str.upper().str.strip())
    .groupby(['gvkey', 'maturity'], as_index=False)
    .agg(start_date=('date', 'min'), end_date=('date', 'max'))
)
gvkey_maturity_windows['horizon_days'] = gvkey_maturity_windows['maturity'].map(maturity_to_horizon)
gvkey_maturity_windows = gvkey_maturity_windows[
    gvkey_maturity_windows['horizon_days'].notna()
].sort_values(['gvkey', 'horizon_days'])

cleaned_cds.to_csv(cleaned_cds_path, index=False)
gvkey_maturity_windows.to_csv(gvkey_maturity_windows_path, index=False)

mapped_companies = mapped['cds_company'].nunique()
total_companies = cds_names['cds_company'].nunique()
coverage = mapped_companies / max(total_companies, 1)

print(f"\nSaved: {cleaned_cds_path} ({len(cleaned_cds):,} rows)")
print(f"Saved: {gvkey_maturity_windows_path} ({len(gvkey_maturity_windows):,} rows)")
print(f"Company mapping coverage: {mapped_companies}/{total_companies} ({coverage:.2%})")
print('\nPreview:')
display(gvkey_maturity_windows.head(20))

In [ ]:
import matplotlib.dates as mdates

windows_path = Path('../data/cds_filters/gvkey_maturity_simulation_windows.csv')
windows = pd.read_csv(windows_path)
windows['gvkey'] = windows['gvkey'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
windows['maturity'] = windows['maturity'].astype(str).str.upper().str.strip()
windows['start_date'] = pd.to_datetime(windows['start_date'], errors='coerce')
windows['end_date'] = pd.to_datetime(windows['end_date'], errors='coerce')

merton_path = Path('../data/output/merged_data_with_merton.csv')
merton = pd.read_csv(merton_path, usecols=['gvkey', 'liabilities_total', 'mkt_cap'])
merton['gvkey'] = merton['gvkey'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
merton['leverage_value'] = merton['liabilities_total'] / merton['mkt_cap'].replace(0, np.nan)

firm_lev = (
    merton.dropna(subset=['leverage_value'])
    .groupby('gvkey', as_index=False)['leverage_value']
    .median()
)

n_groups = 3
quantiles = np.linspace(0, 1, n_groups + 1)
edges = np.unique(firm_lev['leverage_value'].quantile(quantiles).values)
actual = len(edges) - 1
group_names = [f'Leverage Group {i+1} (Low→High)' for i in range(actual)]
edges_adj = edges.copy()
edges_adj[0] -= 1e-12
edges_adj[-1] += 1e-12
firm_lev['leverage_group'] = pd.cut(
    firm_lev['leverage_value'], bins=edges_adj, labels=group_names, include_lowest=True,
)

windows = windows.merge(firm_lev[['gvkey', 'leverage_group', 'leverage_value']], on='gvkey', how='left')
windows = windows.dropna(subset=['leverage_group'])

maturity_label = '5Y'
mat_windows = windows[windows['maturity'] == maturity_label].copy()

all_dates = pd.bdate_range(
    start=mat_windows['start_date'].min(),
    end=mat_windows['end_date'].max(),
)

records = []
for _, row in mat_windows.iterrows():
    mask = (all_dates >= row['start_date']) & (all_dates <= row['end_date'])
    for d in all_dates[mask]:
        records.append({'date': d, 'gvkey': row['gvkey'], 'leverage_group': row['leverage_group']})

daily_panel = pd.DataFrame(records)
firm_counts = (
    daily_panel.groupby(['date', 'leverage_group'])['gvkey']
    .nunique()
    .rename('n_firms')
    .reset_index()
)

colors = {'Leverage Group 1 (Low→High)': 'tab:blue',
          'Leverage Group 2 (Low→High)': 'tab:orange',
          'Leverage Group 3 (Low→High)': 'tab:red'}

fig, axes = plt.subplots(actual, 1, figsize=(14, 3.5 * actual), sharex=True)
if actual == 1:
    axes = [axes]

for ax, gname in zip(axes, group_names):
    sub = firm_counts[firm_counts['leverage_group'] == gname].sort_values('date')
    total_firms = mat_windows[mat_windows['leverage_group'] == gname]['gvkey'].nunique()
    color = colors.get(gname, 'tab:gray')

    ax.fill_between(sub['date'], sub['n_firms'], alpha=0.35, color=color)
    ax.plot(sub['date'], sub['n_firms'], linewidth=1.4, color=color)
    ax.set_ylabel('Number of Firms', fontsize=11)
    ax.set_title(f'{gname}  (total unique firms: {total_firms})', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

axes[-1].set_xlabel('Date', fontsize=11)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].xaxis.set_major_locator(mdates.YearLocator())

fig.suptitle(
    f'Number of Firms with CDS Data Over Time ({maturity_label} Maturity) by Leverage Group',
    fontsize=14, fontweight='bold', y=1.01,
)
fig.tight_layout()
plt.show()

print("\nFirm membership per leverage group:")
for gname in group_names:
    g = firm_lev[firm_lev['leverage_group'] == gname].sort_values('leverage_value')
    matched = g[g['gvkey'].isin(mat_windows['gvkey'])]
    print(f"\n  {gname}  ({len(matched)} firms)")
    for _, r in matched.iterrows():
        print(f"    gvkey={r['gvkey']}  leverage={r['leverage_value']:.4f}")

In [ ]:
maturity_order = ['1Y', '3Y', '5Y']
mat_colors = {'1Y': 'tab:green', '3Y': 'tab:blue', '5Y': 'tab:purple'}

windows_all = pd.read_csv(windows_path)
windows_all['gvkey'] = windows_all['gvkey'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
windows_all['maturity'] = windows_all['maturity'].astype(str).str.upper().str.strip()
windows_all['start_date'] = pd.to_datetime(windows_all['start_date'], errors='coerce')
windows_all['end_date'] = pd.to_datetime(windows_all['end_date'], errors='coerce')

all_dates_mat = pd.bdate_range(
    start=windows_all['start_date'].min(),
    end=windows_all['end_date'].max(),
)

records_mat = []
for _, row in windows_all.iterrows():
    mask = (all_dates_mat >= row['start_date']) & (all_dates_mat <= row['end_date'])
    for d in all_dates_mat[mask]:
        records_mat.append({'date': d, 'gvkey': row['gvkey'], 'maturity': row['maturity']})

daily_panel_mat = pd.DataFrame(records_mat)
firm_counts_mat = (
    daily_panel_mat.groupby(['date', 'maturity'])['gvkey']
    .nunique()
    .rename('n_firms')
    .reset_index()
)

n_mat = len(maturity_order)
fig, axes = plt.subplots(n_mat, 1, figsize=(14, 3.5 * n_mat), sharex=True)
if n_mat == 1:
    axes = [axes]

for ax, mat in zip(axes, maturity_order):
    sub = firm_counts_mat[firm_counts_mat['maturity'] == mat].sort_values('date')
    total_firms = windows_all[windows_all['maturity'] == mat]['gvkey'].nunique()
    color = mat_colors.get(mat, 'tab:gray')

    ax.fill_between(sub['date'], sub['n_firms'], alpha=0.35, color=color)
    ax.plot(sub['date'], sub['n_firms'], linewidth=1.4, color=color)
    ax.set_ylabel('Number of Firms', fontsize=11)
    ax.set_title(f'{mat} Maturity  (total unique firms: {total_firms})', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

axes[-1].set_xlabel('Date', fontsize=11)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].xaxis.set_major_locator(mdates.YearLocator())

fig.suptitle(
    'Number of Firms with CDS Data Over Time by Maturity',
    fontsize=14, fontweight='bold', y=1.01,
)
fig.tight_layout()
plt.show()